In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [16]:
review_df = pd.read_csv("/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")

In [17]:
review_df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [18]:
review_df.shape

(50000, 2)

In [19]:
review_df["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [20]:
review_df.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)

In [21]:
review_df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [11]:
from sklearn.model_selection import train_test_split

In [22]:
train_data, test_data = train_test_split(review_df, test_size=0.2, random_state=12)

In [27]:
train_data.head()

,review,sentiment
35235,Have to disagree with people saying that this ...,0
36936,Husband-and-wife doctor team Carole and Niles ...,1
46486,I like the cast pretty much however the story ...,0
27160,This movie is just so awful. So bad that I can...,0
19490,I purchased the BLOOD CASTLE DVD on eBay for a...,0


In [23]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [24]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_data["review"])

In [25]:
len(tokenizer.word_index)

112162

In [26]:
df_copy = train_data["review"].copy()
reviews_list = df_copy.tolist()
max_word_count = max(len(review.split()) for review in reviews_list)
max_word_count

2470

In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM

In [28]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data["review"])

In [29]:
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)

In [30]:
print(X_train)

[[   0    0    0 ... 2414  438   70]
 [   2 3527   23 ...   24 2139    5]
 [   0    0    0 ...    3  448  448]
 ...
 [ 391    2 1259 ...   15    7    7]
 [   0    0    0 ... 2958    4    1]
 [   0    0    0 ...    5 1615  536]]


In [31]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

**#Model**# 

In [34]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation="sigmoid"))

In [37]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [38]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [39]:
model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 105s 200ms/step - accuracy: 0.7343 - loss: 0.5189 - val_accuracy: 0.8426 - val_loss: 0.3717
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 100s 199ms/step - accuracy: 0.8581 - loss: 0.3427 - val_accuracy: 0.8521 - val_loss: 0.3582
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 100s 200ms/step - accuracy: 0.8828 - loss: 0.2971 - val_accuracy: 0.8257 - val_loss: 0.3915
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 100s 200ms/step - accuracy: 0.8328 - loss: 0.3819 - val_accuracy: 0.8625 - val_loss: 0.3520
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 100s 199ms/step - accuracy: 0.8930 - loss: 0.2659 - val_accuracy: 0.8685 - val_loss: 0.3548


In [40]:
loss, accuracy = model.evaluate(X_test, Y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 23s 73ms/step - accuracy: 0.8705 - loss: 0.3195


In [42]:
print(f"Test Data Loss: {loss}")
print(f"Test Data Accuracy: {accuracy}")

Test Data Loss: 0.3301236033439636
Test Data Accuracy: 0.8669000267982483


In [43]:
import pickle

with open('model.pkl', 'wb') as file:
    pickle.dump(model, file)

# with open('model.pkl', 'rb') as file:
#     loaded_model = pickle.load(file)

In [44]:
def sentiment(review):
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence, maxlen=200)
  prediction = model.predict(padded_sequence)
  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
  return sentiment

In [46]:
review = "Man,I slept during the whole movie"
sentiment(review)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step


'negative'